In [ ]:
import cv2
import numpy as np
import pytesseract
from matplotlib import pyplot as plt
import easyocr

In [ ]:
reader = easyocr.Reader(['en'], gpu=True)

# Reading the image

In [ ]:
# image_path = "../images/1.jpeg"
image_path = "../images/3.jpg"
# image_path = "../images/4.jpg"

img = cv2.imread(image_path)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

In [ ]:
plt.imshow(gray)
plt.show()

In [ ]:
_, binary = cv2.threshold(gray, 128, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)

# plt.imshow(gray)
# plt.show()

# MSER

In [ ]:
mser = cv2.MSER_create(delta=5, min_area=10, max_area=200, max_variation=0.5, min_diversity=0.1)
regions, _ = mser.detectRegions(gray)
# hulls = [cv2.convexHull(p.reshape(-1, 1, 2)) for p in regions]
# cv2.polylines(img, hulls, 1, (0, 255, 0))

# Filter regions by area
filtered_regions = [p for p in regions if len(p) > 10]
# Draw the regions on the image
for p in filtered_regions:
    x, y, w, h = cv2.boundingRect(p.reshape(-1, 1, 2))
    cv2.rectangle(img, (x, y), (x + w, y + h), (255, 0, 0), 2)


In [ ]:
plt.figure(figsize=[8,8])
plt.imshow(img)
plt.show()

In [ ]:
# Pass the preprocessed binary image instead, with a better PSM
config = '--oem 3 --psm 6'
text = pytesseract.image_to_string(binary, config=config)
print(text)

# EAST Text Detection

In [ ]:
model_path = "models/frozen_east_text_detection.pb"  


net = cv2.dnn.readNet(model=model_path)
blob = cv2.dnn.blobFromImage(img, 1.0, (320, 320), (123.68, 116.78, 103.94), True, False)
net.setInput(blob)
scores, geometry = net.forward(["feature_fusion/Conv_7/Sigmoid", "feature_fusion/concat_3"])


In [ ]:
config = '--oem 1 --psm 7'
text = pytesseract.image_to_string(img, config=config)

In [ ]:
text